# Модель Лотки-Вольтерры (Хищник-Жертва)

**Цель работы:** Исследовать колебательную динамику в системе "хищник-жертва".

In [ ]:
using DrWatson
@quickactivate "sir_lv_project"

using DifferentialEquations
using DataFrames
using Plots
using LaTeXStrings
using JLD2

script_name = splitext(basename(PROGRAM_FILE))[1]
mkpath(plotsdir(script_name))
mkpath(datadir(script_name))

## Определение модели
Система уравнений:
$$
\begin{cases}
\frac{dx}{dt} = \alpha x - \beta xy \\
\frac{dy}{dt} = \delta xy - \gamma y
\end{cases}
$$

In [ ]:
function lotka_volterra!(du, u, p, t)
    x, y = u
    α, β, δ, γ = p

    du[1] = α*x - β*x*y
    du[2] = δ*x*y - γ*y
    nothing
end

## Параметры модели

In [ ]:
p = [0.1,   # α - скорость размножения жертв
     0.02,  # β - скорость поедания
     0.01,  # δ - коэффициент конверсии
     0.3]   # γ - смертность хищников

Начальные условия

In [ ]:
u0 = [40.0,  # начальное количество жертв
      9.0]   # начальное количество хищников

tspan = (0.0, 200.0)

## Стационарные точки

In [ ]:
x_star = p[4] / p[3]  # x* = γ/δ
y_star = p[1] / p[2]  # y* = α/β

println("Стационарные точки:")
println("  x* (равновесие жертв) = ", round(x_star, digits=2))
println("  y* (равновесие хищников) = ", round(y_star, digits=2))

## Решение системы

In [ ]:
prob = ODEProblem(lotka_volterra!, u0, tspan, p)
sol = solve(prob, Tsit5(), saveat=0.5)

Создаем DataFrame

In [ ]:
df = DataFrame(t = sol.t)
df.prey = [u[1] for u in sol.u]
df.predator = [u[2] for u in sol.u]

## Визуализация
### Динамика популяций

In [ ]:
p1 = plot(df.t, [df.prey df.predator],
          label=[L"Жертвы (x)" L"Хищники (y)"],
          xlabel="Время",
          ylabel="Численность",
          title="Модель Лотки-Вольтерры",
          linewidth=2,
          color=[:green :red],
          grid=true)

savefig(p1, plotsdir(script_name, "lv_dynamics.png"))

### Фазовый портрет

In [ ]:
p2 = plot(df.prey, df.predator,
          label="Фазовая траектория",
          xlabel="Популяция жертв (x)",
          ylabel="Популяция хищников (y)",
          title="Фазовый портрет",
          color=:blue,
          linewidth=1.5,
          grid=true)

scatter!([x_star], [y_star], color=:black, markersize=8,
         label="Стационарная точка")

savefig(p2, plotsdir(script_name, "lv_phase.png"))

## Анализ колебаний

In [ ]:
function find_periods(t, signal)
    peaks = Float64[]
    peak_times = Float64[]
    for i in 2:length(signal)-1
        if signal[i] > signal[i-1] && signal[i] > signal[i+1]
            push!(peaks, signal[i])
            push!(peak_times, t[i])
        end
    end

    if length(peak_times) >= 2
        periods = [peak_times[i+1] - peak_times[i] for i in 1:length(peak_times)-1]
        return mean(periods), std(periods)
    else
        return NaN, NaN
    end
end

T_prey, _ = find_periods(df.t, df.prey)
println("\nПериод колебаний жертв: ", round(T_prey, digits=2))

Сохранение результатов

In [ ]:
@save datadir(script_name, "lv_results.jld2") df p x_star y_star